In [2]:
from langgraph.graph import   START , END , StateGraph 
from langchain_core.messages.utils import trim_messages , count_tokens_approximately 
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import MemorySaver
from llm import llm
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,BaseMessage,AIMessage
from typing import TypedDict,Annotated 

In [4]:
class MessagesState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]
    summary:str

def chat(state:MessagesState):
    # -----Short term memory using trimming

    # stm=trim_messages(
    #     state['messages'],
    #     strategy="last",
    #     token_counter=count_tokens_approximately,
    #     max_tokens=200
    # )
    
    # -----Short term memory using Summarizing
    msg=[]
    if(state.get('summary')):
        msg.append(AIMessage(content=f"SUmmary of previus conversation : {state['summary']}"))
    msg.extend(state['messages'])

    print("-----------------\n",msg,"\n------------")
    res= llm.invoke(msg).content
    return {"messages":[AIMessage(content=res)]}

In [22]:
def summarize(state:MessagesState):
    if(state.get('summary')):
        prompt=f"Existing summary : {state['summary']} , extend the summary with new conversation"
    else :
        prompt="Summarize the above conversation"

    msg=state['messages'][:4] 
    res=llm.invoke(msg+[HumanMessage(content=prompt)]).content
    print("Summary---------------")
    print(res)
    return {"messages": [RemoveMessage(id=i.id) for i in msg],"summary": res}
def condition(state:MessagesState):
    return len(state['messages'])>6
    

In [23]:
graph=StateGraph(MessagesState)
graph.add_node("chat",chat)
graph.add_node("summary",summarize)

graph.add_edge(START,"chat")
graph.add_conditional_edges("chat",condition,{True:'summary',False:END})
graph.add_edge("chat",END)
checkpoint=MemorySaver()
workflow=graph.compile(checkpointer=checkpoint)

In [24]:
while True:
    user=input("Enter prompt = ")
    print("User - ",user)
    if(user=="1"):
        break
    res=workflow.invoke({'messages':[HumanMessage(content=user)]},config={"configurable":{"thread_id":"1"}})
    print("AI - ",res['messages'][-1].text)

User -  hi
-----------------
 [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='656c9fa7-eb89-4c77-a601-e16e77830c5a')] 
------------
AI -  It's nice to meet you. Is there something I can help you with or would you like to chat?
User -  alex
-----------------
 [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='656c9fa7-eb89-4c77-a601-e16e77830c5a'), AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, id='c706eb47-d9fd-468f-be9c-50180a4af58b', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='alex', additional_kwargs={}, response_metadata={}, id='71aa2c05-32b6-45dc-aeee-b5e34b96b5ee')] 
------------
AI -  Hi Alex. How's your day going so far? Is there something on your mind that you'd like to talk about, or is this just a casual hello?
User -  i am bot
-----------------
 [HumanMessage(content='hi', additional_kwargs={}, respo

In [8]:
workflow.get_state({"configurable":{"thread_id":"1"}}).values['messages']


[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='00ee3f64-2ac3-4168-b074-9c91d02ac97b'),
 AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, id='059525a9-1654-40e5-ae41-47f005f90d5b', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='who are u', additional_kwargs={}, response_metadata={}, id='cada3002-f00c-4e57-926c-9bd074c75649'),
 AIMessage(content="I'm an artificial intelligence (AI) designed to simulate conversations and answer questions to the best of my knowledge. I don't have a personal identity or feelings like humans do, but I'm here to help and provide information on a wide range of topics.\n\nYou can think of me as a virtual assistant, and I'll do my best to:\n\n* Answer your questions\n* Provide information on various subjects\n* Help with language-related tasks\n* Engage in conversation\n* Learn and improve my responses based on our 

In [ ]:
for i in workflow.get_state_history({"configurable":{"thread_id":"1"}}):
    print(i)